In [1]:
import torch
from training_loop import training_loop
from extract_embedding import get_train_test_loaders
from qwen_tts.core.models import BasicSpeakerEncoder
import optuna

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



********
********
 


2026-04-24 12:44:25.371740515 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
# Setup the data loaders
train_loader, test_loader = get_train_test_loaders()


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB
Loading Qwen3-TTS model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 22339.83it/s]


Qwen3 speaker encoder loaded on cuda
train-clean-100 already extracted
dev-clean already extracted
Loaded 12526 samples
Loaded 1953 samples
Train: 12526 samples, 391 batches
Test:  1953 samples, 62 batches


In [3]:

def objective(trial, train_loader, test_loader):
    # 1. Define the hyperparameter search space
    # Optuna will sample values from these ranges for each trial
    num_epochs = trial.suggest_int("num_epochs", 10, 50)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    alpha = trial.suggest_float("loss_alpha", 0.1, 0.9) 
    
    # Assemble the dictionary for the training loop
    hyperparams = {
        "num_epochs": num_epochs,
        "save_every": 20,       # Prevent excessive checkpoint saving during sweeps
        "lr": lr,
        "loss_alpha": alpha,
        "weight_decay": weight_decay,
        "model_name": f"BasicSpeakerEncoder_trial_{trial.number}"
    }

    # 2. Setup Device and model
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    # Setup the model
    model = BasicSpeakerEncoder().to(device)
    
        
    # 3. Execute the training loop
    # We catch the returned best test cosine similarity
    best_test_cosine = training_loop(model, train_loader, test_loader, device, hyperparams)
    
    # Optuna will try to MAXIMIZE this returned value
    return best_test_cosine


In [ ]:
study = optuna.create_study(direction="maximize", study_name="Speaker_Encoder_Distillation")
study.optimize(lambda trial: objective(trial, train_loader, test_loader), n_trials=20)
    
print("\n=================================")
print("Best Trial Found:")
print(f"Best Test Cosine Similarity: {study.best_trial.value:.4f}")
print("Optimal Hyperparameters: ")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-24 12:46:36,942] A new study created in memory with name: Speaker_Encoder_Distillation


Epoch   1/37 | Train loss=0.0808 cos=0.9595 | Test loss=0.0722 cos=0.9750 *best* | 7.6s
Epoch   2/37 | Train loss=0.0692 cos=0.9804 | Test loss=0.0694 cos=0.9801 *best* | 6.8s
Epoch   3/37 | Train loss=0.0672 cos=0.9839 | Test loss=0.0683 cos=0.9822 *best* | 6.8s
Epoch   4/37 | Train loss=0.0665 cos=0.9853 | Test loss=0.0677 cos=0.9833 *best* | 6.9s
Epoch   5/37 | Train loss=0.0659 cos=0.9864 | Test loss=0.0675 cos=0.9836 *best* | 6.9s
Epoch   6/37 | Train loss=0.0655 cos=0.9870 | Test loss=0.0673 cos=0.9839 *best* | 7.0s
Epoch   7/37 | Train loss=0.0653 cos=0.9875 | Test loss=0.0668 cos=0.9848 *best* | 6.9s
Epoch   8/37 | Train loss=0.0650 cos=0.9879 | Test loss=0.0669 cos=0.9846 | 6.9s
Epoch   9/37 | Train loss=0.0648 cos=0.9883 | Test loss=0.0669 cos=0.9846 | 7.0s
Epoch  10/37 | Train loss=0.0647 cos=0.9886 | Test loss=0.0669 cos=0.9846 | 7.0s
Epoch  11/37 | Train loss=0.0645 cos=0.9889 | Test loss=0.0668 cos=0.9848 *best* | 7.1s
Epoch  12/37 | Train loss=0.0643 cos=0.9892 | Test lo

[I 2026-04-24 12:51:07,302] Trial 0 finished with value: 0.9870215462100121 and parameters: {'num_epochs': 37, 'lr': 0.0030822427533699036, 'weight_decay': 0.003763310341530355, 'loss_alpha': 0.5474480172684609}. Best is trial 0 with value: 0.9870215462100121.


Epoch  37/37 | Train loss=0.0626 cos=0.9923 | Test loss=0.0656 cos=0.9870 | 7.6s

Done! Best test cosine similarity: 0.9870
Epoch   1/32 | Train loss=0.1620 cos=0.8085 | Test loss=0.0813 cos=0.9663 *best* | 7.5s
Epoch   2/32 | Train loss=0.0774 cos=0.9739 | Test loss=0.0777 cos=0.9733 *best* | 7.5s
Epoch   3/32 | Train loss=0.0754 cos=0.9777 | Test loss=0.0766 cos=0.9756 *best* | 7.6s
Epoch   4/32 | Train loss=0.0744 cos=0.9796 | Test loss=0.0755 cos=0.9776 *best* | 7.6s
Epoch   5/32 | Train loss=0.0737 cos=0.9810 | Test loss=0.0750 cos=0.9787 *best* | 7.6s
Epoch   6/32 | Train loss=0.0732 cos=0.9820 | Test loss=0.0745 cos=0.9797 *best* | 7.6s
Epoch   7/32 | Train loss=0.0728 cos=0.9828 | Test loss=0.0742 cos=0.9802 *best* | 7.6s
Epoch   8/32 | Train loss=0.0725 cos=0.9834 | Test loss=0.0739 cos=0.9809 *best* | 7.6s
Epoch   9/32 | Train loss=0.0722 cos=0.9839 | Test loss=0.0736 cos=0.9815 *best* | 7.6s
Epoch  10/32 | Train loss=0.0720 cos=0.9844 | Test loss=0.0733 cos=0.9819 *best* | 7

[I 2026-04-24 12:55:13,939] Trial 1 finished with value: 0.9845208804453572 and parameters: {'num_epochs': 32, 'lr': 3.6598211139320726e-05, 'weight_decay': 0.008202242143653452, 'loss_alpha': 0.5032456178470767}. Best is trial 1 with value: 0.9845208804453572.


...

Epoch  16/16 | Train loss=0.0772 cos=0.9912 | Test loss=0.0792 cos=0.9867 *best* | 7.8s

Done! Best test cosine similarity: 0.9867
Epoch   1/26 | Train loss=0.0594 cos=0.9587 | Test loss=0.0461 cos=0.9754 *best* | 8.4s
Epoch   2/26 | Train loss=0.0420 cos=0.9804 | Test loss=0.0424 cos=0.9799 *best* | 7.7s
Epoch   3/26 | Train loss=0.0393 cos=0.9838 | Test loss=0.0410 cos=0.9817 *best* | 7.7s
Epoch   4/26 | Train loss=0.0380 cos=0.9854 | Test loss=0.0395 cos=0.9836 *best* | 7.7s
Epoch   5/26 | Train loss=0.0372 cos=0.9865 | Test loss=0.0392 cos=0.9840 *best* | 8.4s
Epoch   6/26 | Train loss=0.0366 cos=0.9872 | Test loss=0.0390 cos=0.9843 *best* | 7.8s
Epoch   7/26 | Train loss=0.0362 cos=0.9877 | Test loss=0.0388 cos=0.9845 *best* | 7.8s
Epoch   8/26 | Train loss=0.0358 cos=0.9882 | Test loss=0.0387 cos=0.9846 *best* | 7.7s
Epoch   9/26 | Train loss=0.0355 cos=0.9885 | Test loss=0.0385 cos=0.9849 *best* | 7.7s
Epoch  10/26 | Train loss=0.0352 cos=0.9889 | Test loss=0.0382 cos=0.9853 *be

[I 2026-04-24 13:20:38,887] Trial 8 finished with value: 0.9868928693955944 and parameters: {'num_epochs': 26, 'lr': 0.0020270795126449683, 'weight_decay': 6.803373579047638e-06, 'loss_alpha': 0.7952464004996977}. Best is trial 6 with value: 0.9787972848261556.


Epoch  26/26 | Train loss=0.0329 cos=0.9918 | Test loss=0.0369 cos=0.9869 | 7.8s

Done! Best test cosine similarity: 0.9869
Epoch   1/13 | Train loss=0.1493 cos=0.8145 | Test loss=0.0987 cos=0.9658 *best* | 8.0s
Epoch   2/13 | Train loss=0.0959 cos=0.9737 | Test loss=0.0962 cos=0.9731 *best* | 8.0s
Epoch   3/13 | Train loss=0.0946 cos=0.9777 | Test loss=0.0952 cos=0.9761 *best* | 8.1s
Epoch   4/13 | Train loss=0.0939 cos=0.9797 | Test loss=0.0947 cos=0.9775 *best* | 8.1s
Epoch   5/13 | Train loss=0.0935 cos=0.9809 | Test loss=0.0944 cos=0.9785 *best* | 8.0s
Epoch   6/13 | Train loss=0.0932 cos=0.9818 | Test loss=0.0941 cos=0.9795 *best* | 8.0s
Epoch   7/13 | Train loss=0.0930 cos=0.9825 | Test loss=0.0939 cos=0.9802 *best* | 7.9s
Epoch   8/13 | Train loss=0.0928 cos=0.9830 | Test loss=0.0937 cos=0.9806 *best* | 8.0s


[W 2026-04-24 13:21:45,806] Trial 9 failed with parameters: {'num_epochs': 13, 'lr': 3.418018473989304e-05, 'weight_decay': 1.047094985518249e-05, 'loss_alpha': 0.3240872964439736} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_975/2882467666.py", line 2, in <lambda>
    study.optimize(lambda trial: objective(trial, train_loader, test_loader), n_trials=20)
  File "/tmp/ipykernel_975/254313492.py", line 33, in objective
    best_test_cosine = training_loop(model, train_loader, test_loader, device, hyperparams)
  File "/workspace/training_loop.py", line 44, in training_loop
    epoch_loss += loss.item()
KeyboardInterrupt
[W 2026-04-24 13:21:45,809] Trial 9 failed with value None.


KeyboardInterrupt: 